In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
''' 1.added new feature about bedroomarea ratio
    2.added log transform
    3.added cross-validation
    4.added, RfRegressor, LinearRegressor
    5.picked most well perfomed model
    6.added RandomizedSearchCV
    '''

In [ ]:
'''these codes are little(or more) messy but i tried to follow some steps'''

In [ ]:
#read house test file
test=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

#read house test train file
train=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')

In [ ]:
#check info
train.info()

In [ ]:
#drop id for now
train_id=train['Id']
train_1 = train.drop('Id',axis=1)

# 
test_id = test['Id'].copy()    #
test_1 = test.drop(columns=['Id'], axis=1)


In [ ]:
#check how are the numeric histograms
import matplotlib.pyplot as plt
train_1.hist(bins=50, figsize=(12,18))
plt.show()

In [ ]:
#define correlation matrix
corr_matrix=train_1.corr(numeric_only=True)
#check correlation
corr=corr_matrix['SalePrice'].sort_values(ascending=False)

corr

In [ ]:
#leave high corr cols
high_corr_cols=[col for col in corr.index if corr[col]>0.12]

In [ ]:
'''somethings are not really high but ill first devide it in this way and check more
enclosedporch,kichenabvgr,screenporch are greater than 0.1(in absolute)but i can see their actual value in histogram are pretty weird so ill ignore it.'''

In [ ]:
corr[high_corr_cols]

In [ ]:
'''this list isnt so bad but some of them might be related and will affect model repeatedly'''

In [ ]:
'''lets deal with 'bath' features first'''

In [ ]:
#import simple imputer for cateogrical columns
from sklearn.impute import SimpleImputer

#define imputers with different standard
med_imputer=SimpleImputer(strategy='median')
mean_imputer=SimpleImputer(strategy='mean')
mofr_imputer=SimpleImputer(strategy='most_frequent')


In [ ]:
'lets find BsmtHalfBath first'

In [ ]:
train_1['BsmtHalfBath']

In [ ]:
'''there are no nulls in bath cols so i will just replace them with total bath'''

In [ ]:

#combine bath values in cols
train_1['Total_Bath']=train_1['BsmtFullBath']+train_1['FullBath']+train_1['HalfBath']*0.5+train_1['BsmtHalfBath']*0.5

#drop original cols
train_1.drop(['BsmtFullBath','FullBath','HalfBath','BsmtHalfBath'],axis=1,inplace=True)


#do the same thing in test set
test['Total_Bath']=test['BsmtFullBath']+test['FullBath']+test['HalfBath']*0.5+test['BsmtHalfBath']*0.5

#drop original cols
test.drop(['BsmtFullBath','FullBath','HalfBath','BsmtHalfBath'],axis=1,inplace=True)


In [ ]:
'''lets check correlation again'''

In [ ]:
#define correlation matrix
corr_matrix=train_1.corr(numeric_only=True)
#check correlation
corr_2=corr_matrix['SalePrice'].sort_values(ascending=False)

high_corr_cols_2=[col for col in corr_2.index if corr_2[col]>0.12]

corr_2[high_corr_cols_2]

In [ ]:
'''i can see total bath has pretty high correlation even when compared with fullbath '''

In [ ]:
#compare
corr_2['Total_Bath']-corr['FullBath']

In [ ]:
'''i wanna make Total leaving area(interior/exterior/total) for see how is the correlation ill still devide it because model might just learn well itself'''

In [ ]:
#check list of numeric cols
train_1.select_dtypes(include=['int64','float64']).columns

In [ ]:
#define exterior/interior/total
interior_cols=['GrLivArea', 'TotalBsmtSF']
exterior_cols=['GarageArea', 'WoodDeckSF', 'OpenPorchSF',
    'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea' ]

total_inexarea_cols=interior_cols+exterior_cols


In [ ]:
#check nulls in area cols
train_1[total_inexarea_cols].isnull().sum()

In [ ]:
#replace area cols
train_1['interior_area']=train_1[interior_cols].sum(axis=1)
train_1['exterior_area']=train_1[exterior_cols].sum(axis=1)
train_1['Total_Area']=train_1[total_inexarea_cols].sum(axis=1)


#do the same thing in test set
test['interior_area']=test[interior_cols].sum(axis=1)
test['exterior_area']=test[exterior_cols].sum(axis=1)
test['Total_Area']=test[total_inexarea_cols].sum(axis=1)



In [ ]:
#define correlation matrix
corr_matrix=train_1.corr(numeric_only=True)
#check correlation
corr_3=corr_matrix['SalePrice'].sort_values(ascending=False)

high_corr_cols_3=[col for col in corr_3.index if corr_3[col]>0.12]

corr_3[high_corr_cols_3]

In [ ]:
'''total_area which is highly realated with interior,extrior area is highly related with saleprice...'''

In [ ]:
'''lets drop some numeric columns that are containing repeated info or can mess the model'''

In [ ]:
numeric_cols=train_1.select_dtypes(include=['int64','float64']).columns

numeric_cols

In [ ]:
'before drop, lets deal with some columns that has low correlation but can be useful or least trying to make it useful a little'

In [ ]:
#define correlation matrix
corr_matrix=train_1.corr(numeric_only=True)
#check correlation
corr_3=corr_matrix['SalePrice'].sort_values(ascending=False)

corr_low=[col for col in corr_3.index if corr_3[col]<0.12]

corr_3[corr_low]

In [ ]:
#deal with pool area
train_1['PoolArea'].value_counts()

In [ ]:
'''this poolarea its pretty weird and might be not so good to learn'''

In [ ]:
#make new column that shows it has pool or no
train_1['HasPool'] = (train_1['PoolArea'] > 0).astype(int)
test['HasPool'] = (test['PoolArea'] > 0).astype(int)

#drop original column
train_1.drop('PoolArea', axis=1, inplace=True)
test.drop('PoolArea', axis=1, inplace=True)


In [ ]:
'''lets just drop columns that seems uselss....even if im not so sure'''

In [ ]:

#set new column group
train_important_numeric_cols=train_1[[col for col in  high_corr_cols if col not in ['FullBath', 'HalfBath', 'BsmtFullBath','PoolArea']]]

numeric_cols_adjusted=[col for col in numeric_cols if col not in ['FullBath', 'HalfBath', 'BsmtFullBath','PoolArea']]
#drop low correlation columns 
train_numeric_dropped=train_1.drop(numeric_cols_adjusted,axis=1)
#combine categorical columns and high correlation numeric columns
train_2=pd.concat([train_important_numeric_cols,train_numeric_dropped],axis=1)



#set test high correlation cols
test_high_corr_col_3=[col for col in high_corr_cols_3 if col not in['SalePrice' ,'PoolArea']]
#
test_important_numeric_cols=test[test_high_corr_col_3]
#set numeric cols in test set
test_numeric_cols=[col for col in numeric_cols if col not in['SalePrice' , 'PoolArea']]
#
test_numeric_dropped=test.drop(test_numeric_cols,axis=1)
#
test_2=pd.concat([test_important_numeric_cols,test_numeric_dropped],axis=1)

In [ ]:
#check numeric cols in new train set
numeric_cols_2=train_2.select_dtypes(include=['int64','float64']).columns

numeric_cols_2

In [ ]:
#define correlation matrix
corr_matrix=train_2.corr(numeric_only=True)
#check correlation
corr_4=corr_matrix['SalePrice'].sort_values(ascending=False)

corr_high_4=[col for col in corr_4.index if corr_4[col]>0.12]

corr_4[corr_high_4]

In [ ]:
'''some columns have repeated information inside but ill not touch it because model might be working fine'''

In [ ]:
'''lets deal with nulls in numeric cols'''

In [ ]:
#check  nulls
train_2[numeric_cols_2].isnull().sum()

In [ ]:
'''ill set lot frontage to 0 because they just dotn have it and garageyrblt,masvnrarea too for model to learn'''

In [ ]:
train_2[['LotFrontage','GarageYrBlt','MasVnrArea']]=train_2[['LotFrontage','GarageYrBlt','MasVnrArea']].fillna(0)
test[['LotFrontage','GarageYrBlt','MasVnrArea']]=test[['LotFrontage','GarageYrBlt','MasVnrArea']].fillna(0)

In [ ]:
#check  nulls
train_2[numeric_cols_2].isnull().sum()

In [ ]:
'''lets check categorical columns '''

In [ ]:
categorical_columns=[col for col in train_2.columns if col not in numeric_cols_2]

train_2[categorical_columns]

In [ ]:
train_2[categorical_columns].isnull().sum()

In [ ]:
'lets fill nulls with none in categorical columns'

In [ ]:
train_2[categorical_columns]=train_2[categorical_columns].fillna('None')
test[categorical_columns]=test[categorical_columns].fillna('None')

In [ ]:
train_2[categorical_columns].isnull().sum()

In [ ]:
'''qual,QC has some  more values that can be replaced with order'''

In [ ]:
train_2['ExterCond'].value_counts()

In [ ]:
quality_columns=['PoolQC','FireplaceQu','KitchenQual','HeatingQC','BsmtQual','ExterQual','BsmtCond','GarageCond','ExterCond']

In [ ]:
#define how to set orders
quality_map = {'None': 0,'Po': 1,'Fa': 2,'TA': 3,'Gd': 4, 'Ex': 5}
#replace text with ordinal numbers
for col in quality_columns:
    train_2[col] = train_2[col].map(quality_map)

#do it in test set
for col in quality_columns:
    test[col] = test[col].map(quality_map)


In [ ]:
'''centralair should be 0/1 i think'''

In [ ]:
train_2['CentralAir'].value_counts()

In [ ]:
yes_or_no={'Y':1,'N':0}
train_2['CentralAir']=train_2['CentralAir'].map(yes_or_no)

test['CentralAir']=test['CentralAir'].map(yes_or_no)

In [ ]:
'''lets do onehotencoding but before do that i should decide what columns first'''

In [ ]:
#
numeric_cols_3=train_2.select_dtypes(include=['int64','float64']).columns #have to deal with saleprice in test set later

still_categorical_columns= [col for col in train_2.columns if col not in numeric_cols_3]

In [ ]:
#check unique values in categorical columns
train_2[still_categorical_columns].nunique()

In [ ]:
'''lets one hot encode except three columns that has a lot of unique values, but first lets encode these 3 in some way'''

In [ ]:
#replace unique values with their ratio
freq = train["Neighborhood"].value_counts(normalize=True)
train_2["Neighborhood_freq"] = train_2["Neighborhood"].map(freq)

#
test["Neighborhood_freq"] = test["Neighborhood"].map(freq)

In [ ]:
#replace unique values with their ratio
freq = train["Exterior1st"].value_counts(normalize=True)
train_2["Exterior1st_freq"] = train_2["Exterior1st"].map(freq)

#
test["Exterior1st_freq"] = test["Exterior1st"].map(freq)

In [ ]:
#replace unique values with their ratio
freq = train["Exterior2nd"].value_counts(normalize=True)
train_2["Exterior2nd"] = train_2["Exterior2nd"].map(freq)

#
test["Exterior2nd_freq"] = test["Exterior2nd"].map(freq)

In [ ]:
'''lets one hot encode others that has low unique values(im not really sure some are low...)'''

In [ ]:
#fill nan first
train_2.fillna('None')

In [ ]:
#one hot encode
train_3=pd.get_dummies(train_2)
test_2  = pd.get_dummies(test)

train_3, test_2 = train_3.align(test_2, join='left', axis=1, fill_value=0)

In [ ]:
#check
train_3

In [ ]:
train_3.info()

In [ ]:
train_3.select_dtypes(include='number').columns

In [ ]:
train_3['BedroomRatio']=train_3['BedroomAbvGr']/train_3['GrLivArea']
test_2['BedroomRatio']=test_2['BedroomAbvGr']/test_2['GrLivArea']

In [ ]:
#define correlation matrix
corr_matrix=train_3.corr(numeric_only=True)
#check correlation
corr_5=corr_matrix['SalePrice'].sort_values(ascending=False)

corr_high_5=[col for col in corr_5.index if corr_5[col]>0.3]

corr_5[corr_high_5]

In [ ]:
from sklearn.preprocessing import FunctionTransformer

num_cols = train_3.select_dtypes(include='number').columns
positive_cols = [c for c in num_cols if (train_3[c] > 0).all()]
train_3[positive_cols] = np.log(train_3[positive_cols])

test_2[positive_cols] = np.log(test_2[positive_cols])



In [ ]:
import matplotlib.pyplot as plt
train_3.hist(bins=50, figsize=(12,18))
plt.show()

In [ ]:
'''lets put it in a model'''

In [ ]:
from sklearn.model_selection import train_test_split

X = train_3.drop(columns=['SalePrice','Id'], errors='ignore')

y = train_3['SalePrice'].copy()

X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:



from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor


models = [  RandomForestRegressor(random_state=42),
    XGBRegressor(random_state=42)
]

for model in models:
    scores = -cross_val_score(
        model, X_train, y_train,
        scoring='neg_root_mean_squared_error', cv=5
    )
    print(model.__class__.__name__, ":", scores.mean())




In [ ]:


from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint,uniform #randint=integer, uniform=float

param_distribs={'n_estimators':randint(500,2500),'max_depth':randint(5,10),
               'subsample':uniform(0.6,0.9),'colsample_bytree':uniform(0.6,0.9),
                'learning_rate':uniform(0.01,0.03)}

search_XGB=RandomizedSearchCV(
   XGBRegressor(random_state=42),
   param_distributions=param_distribs,
   n_iter=30,scoring='neg_root_mean_squared_error',cv=5,random_state=42,verbose=1)

search_XGB.fit(X_train, y_train)
print("Best params:", search_XGB.best_params_)
print("Best RMSE:", -search_XGB.best_score_)

In [ ]:


best_model = search_XGB.best_estimator_
best_model.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

#
rmsle = np.sqrt(np.mean((np.log1p(preds_exp) - np.log1p(y_valid_exp))**2))
print(f"{rmsle:.5f}")


In [ ]:
#reindex it
X_test = test_2.drop(columns=['SalePrice','Id'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)   

In [ ]:
#check with visual
import matplotlib.pyplot as plt


plt.scatter(y_valid,preds,alpha=0.6)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("XGBoost Predictions vs Actual")
plt.plot([y_valid.min(), y_valid.max()],
         [y_valid.min(), y_valid.max()],
         'r--')
plt.show()

In [ ]:
#make submission
test_pred = np.exp(best_model.predict(X_test))
output = pd.DataFrame({'Id': test_id, 'SalePrice': test_pred})
output.to_csv('submission.csv', index=False)


In [ ]:
'Thank you for read. i really appreciate it.'

In [ ]:
'I realized that actually submission that i did before did mixing with train and test data...this time i didnt so its improvment...'